In [2]:
import re
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import MultinomialNB

# =========================================================
# 1. LOAD DATA
# =========================================================
file_tfidf = "tabel_tfidf_manual_bigram.csv"
file_label = "hasil_labeling.csv"

df_tfidf = pd.read_csv(file_tfidf, sep=';', low_memory=False)
df_label = pd.read_csv(file_label, sep=';')

# =========================================================
# 2. AMBIL KOLOM BOBOT TF-IDF
# =========================================================
weight_cols = [col for col in df_tfidf.columns if re.match(r"^W_D\d+$", col)]

if len(weight_cols) == 0:
    raise ValueError("Kolom W_D1, W_D2, ... tidak ditemukan.")

# =========================================================
# 3. FUNGSI KONVERSI NILAI NUMERIK
# =========================================================
def parse_number(x):
    if pd.isna(x):
        return 0.0

    s = str(x).strip()
    if s == "":
        return 0.0

    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    elif s.count(".") > 1:
        first, rest = s.split(".", 1)
        s = first + "." + rest.replace(".", "")

    try:
        return float(s)
    except:
        return 0.0

# =========================================================
# 4. BENTUK MATRIKS FITUR DAN LABEL
#    Bentuk akhir: dokumen x fitur
# =========================================================
X = df_tfidf[weight_cols].apply(lambda col: col.map(parse_number)).T.values
y = df_label["label"].values

if len(y) != X.shape[0]:
    raise ValueError(
        f"Jumlah label ({len(y)}) tidak sama dengan jumlah dokumen ({X.shape[0]})."
    )

print("Shape X:", X.shape)
print("Jumlah label:", len(y))

# =========================================================
# 5. SPLIT DATA 80:20
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Jumlah data latih:", X_train.shape[0])
print("Jumlah data uji  :", X_test.shape[0])

# simpan split untuk tahap evaluasi
np.savez_compressed(
    "split_data.npz",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

# =========================================================
# 6. MODELING NAIVE BAYES
# =========================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_nb = MultinomialNB(alpha=1.0)

cv_scores_nb = cross_val_score(
    model_nb,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

mean_cv_nb = cv_scores_nb.mean()

# latih model final NB
model_nb.fit(X_train, y_train)

# simpan model
joblib.dump(model_nb, "model_naive_bayes.pkl")

# simpan hasil modeling NB
hasil_nb = pd.DataFrame([{
    "model": "Naive Bayes",
    "alpha": 1.0,
    "cv_fold_1": cv_scores_nb[0],
    "cv_fold_2": cv_scores_nb[1],
    "cv_fold_3": cv_scores_nb[2],
    "cv_fold_4": cv_scores_nb[3],
    "cv_fold_5": cv_scores_nb[4],
    "mean_cv_accuracy": mean_cv_nb
}])

hasil_nb.to_csv("hasil_modeling_naive_bayes.csv", index=False)

print("\n=== HASIL MODELING NAIVE BAYES ===")
print(hasil_nb)

# =========================================================
# 7. MODELING NAIVE BAYES + PSO
# =========================================================
def fitness_function(alpha_particles, X_train, y_train, cv):
    fitness_values = []

    for alpha in alpha_particles:
        alpha = max(float(alpha), 1e-5)

        model = MultinomialNB(alpha=alpha)

        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="accuracy"
        )

        mean_acc = scores.mean()
        fitness_values.append(-mean_acc)

    return np.array(fitness_values)

class SimplePSO:
    def __init__(
        self,
        n_particles=8,
        n_iterations=10,
        lower_bound=0.0001,
        upper_bound=5.0,
        w=0.7,
        c1=1.5,
        c2=1.5,
        random_state=42
    ):
        self.n_particles = n_particles
        self.n_iterations = n_iterations
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        self.w = w
        self.c1 = c1
        self.c2 = c2
        self.random_state = random_state
        np.random.seed(self.random_state)

    def optimize(self, X_train, y_train, cv):
        positions = np.random.uniform(
            self.lower_bound,
            self.upper_bound,
            self.n_particles
        )

        velocities = np.random.uniform(-0.1, 0.1, self.n_particles)

        pbest_positions = positions.copy()
        pbest_scores = fitness_function(positions, X_train, y_train, cv)

        gbest_index = np.argmin(pbest_scores)
        gbest_position = pbest_positions[gbest_index]
        gbest_score = pbest_scores[gbest_index]

        history = []

        for iteration in range(self.n_iterations):
            r1 = np.random.rand(self.n_particles)
            r2 = np.random.rand(self.n_particles)

            velocities = (
                self.w * velocities
                + self.c1 * r1 * (pbest_positions - positions)
                + self.c2 * r2 * (gbest_position - positions)
            )

            positions = positions + velocities
            positions = np.clip(positions, self.lower_bound, self.upper_bound)

            current_scores = fitness_function(positions, X_train, y_train, cv)

            improved = current_scores < pbest_scores
            pbest_positions[improved] = positions[improved]
            pbest_scores[improved] = current_scores[improved]

            best_idx = np.argmin(pbest_scores)
            if pbest_scores[best_idx] < gbest_score:
                gbest_score = pbest_scores[best_idx]
                gbest_position = pbest_positions[best_idx]

            history.append({
                "iterasi": iteration + 1,
                "alpha_terbaik": float(gbest_position),
                "cv_accuracy_terbaik": float(-gbest_score)
            })

            print(
                f"Iterasi {iteration+1:02d} | "
                f"alpha terbaik = {gbest_position:.6f} | "
                f"CV accuracy = {-gbest_score:.4f}"
            )

        return gbest_position, -gbest_score, pd.DataFrame(history)

pso = SimplePSO(
    n_particles=8,
    n_iterations=10,
    lower_bound=0.0001,
    upper_bound=5.0,
    w=0.7,
    c1=1.5,
    c2=1.5,
    random_state=42
)

best_alpha, best_cv_acc, history_pso = pso.optimize(X_train, y_train, cv)

# model final NB + PSO
model_nb_pso = MultinomialNB(alpha=best_alpha)
model_nb_pso.fit(X_train, y_train)

# simpan model
joblib.dump(model_nb_pso, "model_naive_bayes_pso.pkl")

# simpan riwayat iterasi PSO
history_pso.to_csv("riwayat_pso.csv", index=False)

# simpan hasil modeling NB + PSO
hasil_nb_pso = pd.DataFrame([{
    "model": "Naive Bayes + PSO",
    "alpha_terbaik": best_alpha,
    "mean_cv_accuracy": best_cv_acc
}])

hasil_nb_pso.to_csv("hasil_modeling_naive_bayes_pso.csv", index=False)

print("\n=== HASIL MODELING NAIVE BAYES + PSO ===")
print(hasil_nb_pso)

print("\nFile output modeling:")
print("1. hasil_modeling_naive_bayes.csv")
print("2. hasil_modeling_naive_bayes_pso.csv")
print("3. riwayat_pso.csv")
print("4. model_naive_bayes.pkl")
print("5. model_naive_bayes_pso.pkl")
print("6. split_data.npz")

Shape X: (1127, 1540)
Jumlah label: 1127
Jumlah data latih: 901
Jumlah data uji  : 226

=== HASIL MODELING NAIVE BAYES ===
         model  alpha  cv_fold_1  cv_fold_2  cv_fold_3  cv_fold_4  cv_fold_5  \
0  Naive Bayes    1.0    0.40884   0.422222   0.444444   0.444444       0.45   

   mean_cv_accuracy  
0           0.43399  
Iterasi 01 | alpha terbaik = 4.753576 | CV accuracy = 0.4817
Iterasi 02 | alpha terbaik = 5.000000 | CV accuracy = 0.4828
Iterasi 03 | alpha terbaik = 5.000000 | CV accuracy = 0.4828
Iterasi 04 | alpha terbaik = 5.000000 | CV accuracy = 0.4828
Iterasi 05 | alpha terbaik = 5.000000 | CV accuracy = 0.4828
Iterasi 06 | alpha terbaik = 5.000000 | CV accuracy = 0.4828
Iterasi 07 | alpha terbaik = 5.000000 | CV accuracy = 0.4828
Iterasi 08 | alpha terbaik = 5.000000 | CV accuracy = 0.4828
Iterasi 09 | alpha terbaik = 5.000000 | CV accuracy = 0.4828
Iterasi 10 | alpha terbaik = 5.000000 | CV accuracy = 0.4828

=== HASIL MODELING NAIVE BAYES + PSO ===
               model